<a href="https://colab.research.google.com/github/TejaSree023/self-pruning-neural-network/blob/main/self_pruning_neural_network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt

In [13]:
# DEVICE
def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [12]:
# PRUNABLE LINEAR
class PrunableLinear(nn.Module):
    def __init__(self, in_features, out_features, bias=True):
        super().__init__()

        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.02)
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None

        # Balanced initialization
        self.gate_scores = nn.Parameter(torch.randn_like(self.weight) - 1)

        # Moderate temperature
        self.temperature = 0.5

    def forward(self, x):
        gates = torch.sigmoid(self.gate_scores / self.temperature)
        pruned_weights = self.weight * gates
        return F.linear(x, pruned_weights, self.bias)

    def get_gates(self):
        return torch.sigmoid(self.gate_scores / self.temperature)

In [14]:
# MODEL

class PrunableMLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.layers = nn.Sequential(
            PrunableLinear(3072, 1024),
            nn.ReLU(),
            nn.Dropout(0.2),

            PrunableLinear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.2),

            PrunableLinear(512, 256),
            nn.ReLU(),

            PrunableLinear(256, 10)
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.layers(x)

    def get_all_gates(self):
        return [m.get_gates() for m in self.modules() if isinstance(m, PrunableLinear)]

In [15]:
# LOSS (SPEC-COMPLIANT)
def compute_loss(model, outputs, targets, lambda_sparsity):
    ce_loss = F.cross_entropy(outputs, targets)

    total_sum = 0.0
    total_elems = 0

    for gates in model.get_all_gates():
        total_sum += gates.sum()
        total_elems += gates.numel()

    # L1-style but normalized for stability
    sparsity_loss = total_sum / total_elems

    total_loss = ce_loss + lambda_sparsity * sparsity_loss
    return total_loss, ce_loss, sparsity_loss

In [16]:
# METRICS
def compute_sparsity(model, threshold=1e-2):
    total = 0
    zero = 0

    for gates in model.get_all_gates():
        total += gates.numel()
        zero += (gates < threshold).sum().item()

    return 100.0 * zero / total

def accuracy(outputs, targets):
    preds = outputs.argmax(dim=1)
    return (preds == targets).float().mean().item()

In [17]:
# TRAIN
def train(model, loader, optimizer, device, lambda_sparsity):
    model.train()

    total_loss, total_acc = 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        outputs = model(x)
        loss, _, _ = compute_loss(model, outputs, y, lambda_sparsity)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_acc += accuracy(outputs, y)

    return total_loss / len(loader), total_acc / len(loader)


In [18]:
# EVAL
def evaluate(model, loader, device):
    model.eval()
    total_acc = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            total_acc += accuracy(outputs, y)

    return total_acc / len(loader)

In [19]:
# DATA
def get_dataloaders(batch_size=128):
    transform = transforms.Compose([
        transforms.ToTensor()
    ])

    train_data = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
    test_data = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=batch_size)

    return train_loader, test_loader

In [20]:
# PLOT
def plot_gate_distribution(model):
    all_gates = []

    for gates in model.get_all_gates():
        all_gates.append(gates.detach().cpu().numpy().flatten())

    all_gates = np.concatenate(all_gates)

    plt.figure()
    plt.hist(all_gates, bins=50)
    plt.title("Gate Value Distribution")
    plt.xlabel("Gate value")
    plt.ylabel("Frequency")
    plt.show()

In [21]:
# RUN
def run_experiment(lambda_sparsity):
    device = get_device()
    model = PrunableMLP().to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    train_loader, test_loader = get_dataloaders()

    for epoch in range(10):
        train_loss, train_acc = train(model, train_loader, optimizer, device, lambda_sparsity)
        test_acc = evaluate(model, test_loader, device)
        sparsity = compute_sparsity(model)

        print(f"[λ={lambda_sparsity}] Epoch {epoch+1}")
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Test Acc: {test_acc:.4f} | Sparsity: {sparsity:.2f}%")
        print("-" * 50)

    return model, test_acc, sparsity

In [ ]:
#main
if __name__ == "__main__":
    lambdas = [1e-4, 5e-4, 1e-3]

    results = []
    best_model = None
    best_acc = 0

    for lam in lambdas:
        model, acc, sp = run_experiment(lam)
        results.append((lam, acc, sp))

        if acc > best_acc:
            best_acc = acc
            best_model = model

    print("\nFinal Results:")
    for lam, acc, sp in results:
        print(f"Lambda: {lam} | Accuracy: {acc:.4f} | Sparsity: {sp:.2f}%")

    # Plot best model gate distribution
    plot_gate_distribution(best_model)

[λ=0.0001] Epoch 1
Train Loss: 2.0194 | Train Acc: 0.2374
Test Acc: 0.3131 | Sparsity: 9.75%
--------------------------------------------------
[λ=0.0001] Epoch 2
Train Loss: 1.8126 | Train Acc: 0.3399
Test Acc: 0.3740 | Sparsity: 9.77%
--------------------------------------------------
[λ=0.0001] Epoch 3
Train Loss: 1.7165 | Train Acc: 0.3791
Test Acc: 0.4134 | Sparsity: 9.80%
--------------------------------------------------
[λ=0.0001] Epoch 4
Train Loss: 1.6448 | Train Acc: 0.4096
Test Acc: 0.4299 | Sparsity: 9.83%
--------------------------------------------------
[λ=0.0001] Epoch 5
Train Loss: 1.5933 | Train Acc: 0.4270
Test Acc: 0.4499 | Sparsity: 9.85%
--------------------------------------------------
[λ=0.0001] Epoch 6
Train Loss: 1.5538 | Train Acc: 0.4451
Test Acc: 0.4668 | Sparsity: 9.88%
--------------------------------------------------


## Why L1 Regularization Encourages Sparsity

We apply an L1-style penalty on the gate values. L1 regularization applies a constant gradient pressure toward zero, unlike L2 which shrinks values smoothly. Since gate values control whether a weight is active (≈1) or pruned (≈0), minimizing the sum of gate values encourages many gates to become very small. As a result, the network learns a sparse structure by effectively disabling less important connections during training.